In [1]:
packages = ["sklearn", "mlflow", "shap", "mlxtend"]

for pkg in packages:
    try:
        module = __import__(pkg)
        version = getattr(module, "__version__", "unknown version")
        print(f"{pkg}: installed ({version})")
    except ImportError:
        print(f"{pkg}: NOT installed")

sklearn: installed (1.7.2)
mlflow: NOT installed
shap: NOT installed
mlxtend: NOT installed


In [1]:
packages = ["sklearn", "mlflow", "shap", "mlxtend"]

for pkg in packages:
    try:
        module = __import__(pkg)
        version = getattr(module, "__version__", "unknown version")
        print(f"{pkg}: installed ({version})")
    except ImportError:
        print(f"{pkg}: NOT installed")

sklearn: installed (1.9.1)
mlflow: installed (3.16.0)
shap: installed (0.52.0)
mlxtend: installed (0.25.0)


### Part 3 Setup: Reuse Part 2's Pipeline and Feature Engineering

Part 3 builds directly on Part 2's cleaned, feature-engineered dataset rather than
reloading/recleaning the raw CSV independently — same reuse pattern as
`visualizations.py` and `traffic_cli.py` importing from `pipeline.py` and
`feature_engineering.py`, just now reaching one directory further up from
`part3_machine_learning/notebooks/`.

In [2]:
import sys
from pathlib import Path

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

PROJECT_ROOT = NOTEBOOK_DIR.parent.parent  # notebooks -> part3_machine_learning -> project root
PART2_DIR = PROJECT_ROOT / "part2_python"
sys.path.insert(0, str(PART2_DIR))

from pipeline import run_pipeline, DATA_FILE
from feature_engineering import engineer_features

clean_df = run_pipeline(DATA_FILE)
feat_df = engineer_features(clean_df)
feat_df.shape

Standardised casing to lowercase in 1730 rows of 'weather_description' (merging case-variant duplicates such as 'Sky is Clear' / 'sky is clear').
Removed 17 exact duplicate rows.
Collapsed 5430 duplicate-timestamp hours (7612 rows removed) down to one row per hour; 0 of these hours had inconsistent traffic_volume across duplicates.
Found 10 rows with physically impossible 'temp' readings (<= 0 Kelvin).
Found 1 rows with physically implausible 'rain_1h' readings (> 1000mm/hour): [9831.3]
Imputed 4 missing 'temp' value(s) in month 1 using that month's median (265.48).
Imputed 6 missing 'temp' value(s) in month 2 using that month's median (265.89).
Imputed 1 missing 'rain_1h' value(s) in month 7 using that month's median (0.00).


(40575, 29)

### Proxy Accident-Risk Label

No accident dataset exists, so we build a documented proxy `high_risk` label per the brief.

**`SEVERE_WEATHER`** reuses the exact same definition as Part 2's `is_severe_weather` flag
(`Thunderstorm`, `Squall`), for consistency across the whole project.

**`is_low_visibility`** is new: weather conditions that impair visibility —
`Fog`, `Mist`, `Haze`, `Smoke`.

**Congestion bucketing for this label uses 4 categories** (Low/Medium/High/Severe, split on
the 25th/50th/75th percentiles), which is different from Part 2's `congestion_category`
(3 categories, split on 25th/75th only). To avoid overwriting or confusing Part 2's column,
this one is named `risk_congestion_category` and lives only in Part 3.

A row is `high_risk` when High/Severe congestion coincides with severe or low-visibility
weather.

In [3]:
import numpy as np

SEVERE_WEATHER = ["Thunderstorm", "Squall"]
LOW_VISIBILITY_CONDITIONS = ["Fog", "Mist", "Haze", "Smoke"]

feat_df["is_low_visibility"] = feat_df["weather_main"].isin(LOW_VISIBILITY_CONDITIONS).astype(int)

# Congestion category (data-driven quartiles of traffic_volume) — 4 buckets, used only
# to construct the proxy risk label, distinct from Part 2's 3-bucket congestion_category
q1, q2, q3 = feat_df["traffic_volume"].quantile([0.25, 0.5, 0.75]).values

def bucket(v):
    if v <= q1:
        return "Low"
    elif v <= q2:
        return "Medium"
    elif v <= q3:
        return "High"
    return "Severe"

feat_df["risk_congestion_category"] = feat_df["traffic_volume"].apply(bucket)

# Proxy accident-risk label
high_congestion = feat_df["risk_congestion_category"].isin(["High", "Severe"])
risky_weather = (
    feat_df["weather_main"].isin(SEVERE_WEATHER)
    | (feat_df["is_low_visibility"] == 1)
)
feat_df["high_risk"] = (high_congestion & risky_weather).astype(int)

print(feat_df["risk_congestion_category"].value_counts())
print()
print(feat_df["is_low_visibility"].value_counts())
print()
print(feat_df["high_risk"].value_counts())
print(feat_df["high_risk"].value_counts(normalize=True))

risk_congestion_category
High      10145
Medium    10144
Low       10144
Severe    10142
Name: count, dtype: int64

is_low_visibility
0    36524
1     4051
Name: count, dtype: int64

high_risk
0    38649
1     1926
Name: count, dtype: int64
high_risk
0    0.952532
1    0.047468
Name: proportion, dtype: float64


### Completing the Common Feature Set for Task 1

Two pieces the brief requires that we don't have yet:
- **Holiday flag**: `holiday` is currently NaN for non-holidays and a holiday's name (e.g. 'Labor Day') otherwise — not usable as a numeric model input directly. We convert it to a clean binary `is_holiday`.
- **Cyclical encoding of day of week**: Part 2 only cyclically encoded hour (`hour_sin`/`hour_cos`). Adding `dow_sin`/`dow_cos` the same way (period 7 instead of 24) captures that Sunday (6) and Monday (0) are adjacent, not far apart.

In [4]:
feat_df["is_holiday"] = feat_df["holiday"].notna().astype(int)

feat_df["dow_sin"] = np.sin(2 * np.pi * feat_df["day_of_week"] / 7)
feat_df["dow_cos"] = np.cos(2 * np.pi * feat_df["day_of_week"] / 7)

print(feat_df["is_holiday"].value_counts())
print(feat_df[["day_of_week", "dow_sin", "dow_cos"]].drop_duplicates().sort_values("day_of_week"))

is_holiday
0    40522
1       53
Name: count, dtype: int64
     day_of_week   dow_sin   dow_cos
126            0  0.000000  1.000000
0              1  0.781831  0.623490
15             2  0.974928 -0.222521
35             3  0.433884 -0.900969
59             4 -0.433884 -0.900969
81             5 -0.974928 -0.222521
104            6 -0.781831  0.623490
